# MDD DIRECT — resampled data QCVerifies that all 3525 subjects came through `02_resample_to_fmri_grid.py` correctly.Expected: **3525 subjects x 5 modalities = 17,625 files**, each `53x63x52` float32 on the`Np` grid — affine `diag(-3,3,3)`, offset `(78,-112,-70)`, `sform_code` 4.**Designed to run on the cluster**, where `/data/users2/...` is a local filesystem.Section 4 opens every output file (~5 GB); that is well under a minute locally and hoursover sshfs.Sections 1-3 give a verdict without reading any voxel data — run those first. Everysection can be run on its own after section 0; nothing depends on an earlier sectionexcept 8, which needs 7.Override any path without editing cells:```bashexport MDD_PROC=/data/users2/ppopov1/datasets/MDD_DIRECT_procexport MDD_PATHS=/somewhere/mdd_direct_volume_paths.csvexport MDD_CACHE=/somewhere/qc_cache.csv```

## 0. Setup — run this first

In [ ]:
%matplotlib inlineimport os, timefrom pathlib import Pathfrom collections import Counterfrom concurrent.futures import ThreadPoolExecutorimport numpy as npimport pandas as pdimport matplotlib.pyplot as plttry:    import nibabel as nibexcept ImportError as e:    raise ImportError(        "nibabel is required. Activate the env that has it, e.g.\n"        "  source /data/users2/ppopov1/miniconda/bin/activate pile\n"        "or:  pip install --user nibabel") from e# ---------------------------------------------------------------- pathsPROC = Path(os.environ.get("MDD_PROC", "/data/users2/ppopov1/datasets/MDD_DIRECT_proc"))def _find_paths_csv():    """The CSV from 01_scan_mdd_direct.py, wherever the repo happens to sit."""    cands = []    if os.environ.get("MDD_PATHS"):        cands.append(Path(os.environ["MDD_PATHS"]))    here = Path.cwd()    for base in (here, here.parent, here.parent.parent):        cands += [base / "data" / "mdd_direct_volume_paths.csv",                  base / "docs" / "03_multimod" / "data" / "mdd_direct_volume_paths.csv"]    cands.append(PROC / "mdd_direct_volume_paths.csv")    return next((c for c in cands if c.exists()), None)PATHS = _find_paths_csv()# cache goes beside the data, which is always writable, not into the repoCACHE = Path(os.environ.get("MDD_CACHE", PROC / "qc_cache.csv"))# ---------------------------------------------------------------- constantsMODALITIES = ["GM_probseg", "WM_probseg", "CSF_probseg", "fALFF", "fALFF_globalC"]PROBSEG    = [m for m in MODALITIES if m.endswith("probseg")]FALFF      = [m for m in MODALITIES if m.startswith("fALFF")]CHUNK_SIZE = 100          # must match what 02_submit_resample.sh usedEXPECTED_SHAPE  = (53, 63, 52)EXPECTED_AFFINE = np.array([[-3.,0,0,78.],[0,3.,0,-112.],[0,0,3.,-70.],[0,0,0,1.]])EXPECTED_SFORM  = 4# ---------------------------------------------------------------- subjects# Prefer the CSV: it is the only source that knows about subjects which produced# no output at all. Falling back to the logs still catches attempted-and-failed.# Falling back to the directory can only see successes -- it cannot detect gaps.chunk_logs = sorted(PROC.glob("resample_log_chunk*.csv")) if PROC.is_dir() else []if PATHS is not None:    subjects = pd.read_csv(PATHS)["subject_id"].astype(str).tolist()    SUBJ_SRC = f"CSV {PATHS}"elif chunk_logs:    subjects = sorted({s for f in chunk_logs                       for s in pd.read_csv(f)["subject_id"].astype(str)})    SUBJ_SRC = f"{len(chunk_logs)} chunk logs"    print("WARNING: no paths CSV found; subject list taken from the logs.")else:    subjects = sorted({p.name.split("_")[0] for p in PROC.glob("*_3mm.nii.gz")})    SUBJ_SRC = "output directory"    print("WARNING: no paths CSV and no logs. Subject list derived from files that\n"          "         EXIST, so missing subjects are undetectable. Set $MDD_PATHS.")# ---------------------------------------------------------------- helpersdef out_path(sid, mod):    return PROC / f"{sid}_{mod}_3mm.nii.gz"def load(sid, mod):    return np.asanyarray(nib.load(str(out_path(sid, mod))).dataobj, dtype=np.float32)def robust_z(x):    med = np.median(x); mad = np.median(np.abs(x - med))    return (x - med) / (1.4826 * mad) if mad > 0 else np.zeros_like(np.asarray(x, float))def ortho(ax_row, vol, title, cmap="gray", vmax=None):    """Three mid-slices of a 3D volume into a row of three axes."""    i, j, k = [s // 2 for s in vol.shape]    if vmax is None:        vmax = np.percentile(vol[vol != 0], 99) if (vol != 0).any() else 1.0    for ax, sl in zip(ax_row, [vol[i,:,:].T, vol[:,j,:].T, vol[:,:,k].T]):        ax.imshow(sl, origin="lower", cmap=cmap, vmin=0, vmax=vmax)        ax.set_xticks([]); ax.set_yticks([])    ax_row[0].set_ylabel(title, fontsize=9)# ---------------------------------------------------------------- preflightprint(f"data dir     : {PROC}"        + ("" if PROC.is_dir() else "   *** NOT FOUND ***"))print(f"paths CSV    : {PATHS or 'not found'}")print(f"cache        : {CACHE}")print(f"subjects     : {len(subjects)} (from {SUBJ_SRC})")print(f"chunk logs   : {len(chunk_logs)}")print(f"expecting    : {len(subjects) * len(MODALITIES)} files")print(f"cwd          : {Path.cwd()}")assert PROC.is_dir(), f"{PROC} not found -- set $MDD_PROC"try:    CACHE.parent.mkdir(parents=True, exist_ok=True)    t = CACHE.parent / ".qc_write_test"; t.touch(); t.unlink()    print("cache dir    : writable")except Exception as e:    print(f"cache dir    : NOT WRITABLE ({e}) -- set $MDD_CACHE")

## 1. Processing logsA chunk that produced no log at all means the array task died before writing. That isinvisible in file counts, so it is the first thing to check.

In [ ]:
log = (pd.concat([pd.read_csv(f) for f in chunk_logs], ignore_index=True)       if chunk_logs else pd.DataFrame(columns=["subject_id","modality","status"]))n_chunks = -(-len(subjects) // CHUNK_SIZE)have = {int(f.stem.split("chunk")[-1]) for f in chunk_logs}missing_chunks = sorted(set(range(n_chunks)) - have)print(f"{len(chunk_logs)} of {n_chunks} chunk logs present")if missing_chunks:    print(f"MISSING CHUNKS: {missing_chunks}")    print(f"  resubmit:  ./02_submit_resample.sh --array {','.join(map(str, missing_chunks))}")if log.empty:    print("\nno log rows -- did the array run?")    gaps, bad = subjects, logelse:    display(log["status"].value_counts().rename("n").to_frame())    covered = set(log["subject_id"].astype(str))    gaps = [s for s in subjects if s not in covered]    print(f"subjects with no log row: {len(gaps)}")    if gaps:        idx = {s: i for i, s in enumerate(subjects)}        print(f"  chunks to resubmit: {sorted({idx[s] // CHUNK_SIZE for s in gaps})}")    bad = log[~log["status"].isin(["written", "skipped_exists"])]    print(f"non-success rows: {len(bad)}")    if len(bad):        display(bad[["subject_id","modality","status"]].head(20))

## 2. File inventoryOne directory listing, then set arithmetic. Opens nothing.

In [ ]:
on_disk = {p.name for p in PROC.glob("*_3mm.nii.gz")}print(f"{len(on_disk)} files on disk\n")inv = pd.DataFrame([    dict(modality=m,         expected=len(subjects),         present=len({f"{s}_{m}_3mm.nii.gz" for s in subjects} & on_disk),         missing=len(subjects) - len({f"{s}_{m}_3mm.nii.gz" for s in subjects} & on_disk))    for m in MODALITIES])display(inv)extra = on_disk - {f"{s}_{m}_3mm.nii.gz" for s in subjects for m in MODALITIES}stray = list(PROC.glob("*.part.nii.gz"))print(f"unexpected files : {len(extra)}  {sorted(extra)[:5] if extra else ''}")print(f"stray .part files: {len(stray)}   (interrupted writes)")for m in MODALITIES:    miss = sorted({s for s in subjects if f"{s}_{m}_3mm.nii.gz" not in on_disk})    if miss:        print(f"  {m}: {len(miss)} missing, e.g. {miss[:5]}")

## 3. Header auditGeometry must be identical across every file. nibabel is lazy, so this reads 352-byteheaders and no voxel data.

In [ ]:
sig, examples = Counter(), {}for s in subjects:    for m in MODALITIES:        p = out_path(s, m)        if not p.exists():            continue        h = nib.load(str(p))        k = (tuple(h.shape), str(h.get_data_dtype()),             tuple(np.round(h.affine.ravel(), 4)), int(h.header["sform_code"]))        sig[k] += 1        examples.setdefault(k, []).append(f"{s}/{m}")print(f"{len(sig)} distinct header signature(s)\n")for k, n in sig.most_common():    shape, dt, aff, sform = k    A = np.array(aff).reshape(4, 4)    ok = (shape == EXPECTED_SHAPE and dt == "float32"          and np.allclose(A, EXPECTED_AFFINE) and sform == EXPECTED_SFORM)    print(f"{'OK ' if ok else 'BAD'}  n={n:6d}  shape={shape}  dtype={dt}  sform={sform}")    print(f"       diag={A.diagonal()[:3]}  offset={A[:3,3]}")    if not ok:        print(f"       examples: {examples[k][:5]}")

## 4. Bulk statistics (cached)Opens every file once. ~30-60 s on the cluster with 8 threads; hours over a remote mount.Cached to `qc_cache.csv` beside the data — set `FORCE = True` to recompute.`mass = sum(voxels) * 27 mm^3`. For probsegs that is tissue volume; for fALFF it has nophysical meaning but still serves as a per-subject scalar for outlier hunting.

In [ ]:
FORCE   = FalseWORKERS = 8LIMIT   = None        # e.g. 200 to trial it on a subsetdef stats_one(job):    sid, mod = job    p = out_path(sid, mod)    if not p.exists():        return dict(subject_id=sid, modality=mod, ok=False, error="missing")    try:        d  = np.asanyarray(nib.load(str(p)).dataobj, dtype=np.float32)        nz = d[d != 0]        return dict(subject_id=sid, modality=mod, ok=True, error="",                    n_nonzero=int(nz.size), mass=float(d.sum()) * 27.0,                    mean_nz=float(nz.mean()) if nz.size else 0.0,                    std_nz=float(nz.std())  if nz.size else 0.0,                    vmin=float(d.min()), vmax=float(d.max()),                    n_nan=int(np.isnan(d).sum()), n_neg=int((d < 0).sum()))    except Exception as e:        return dict(subject_id=sid, modality=mod, ok=False, error=str(e))if CACHE.exists() and not FORCE:    qc = pd.read_csv(CACHE)    print(f"loaded cache: {len(qc)} rows from {CACHE}")else:    subs = subjects[:LIMIT] if LIMIT else subjects    jobs = [(s, m) for s in subs for m in MODALITIES]    t0 = time.time()    with ThreadPoolExecutor(WORKERS) as ex:        qc = pd.DataFrame(list(ex.map(stats_one, jobs)))    qc.to_csv(CACHE, index=False)    print(f"computed {len(qc)} rows in {time.time()-t0:.0f}s -> {CACHE}")qc["ok"] = qc["ok"].fillna(False).astype(bool)print(f"unreadable: {(~qc['ok']).sum()}")g = qc[qc["ok"]]display(g.groupby("modality")[["n_nonzero","mass","mean_nz","vmin","vmax"]]         .agg(["mean","std","min","max"]).round(3))

In [ ]:
# fALFF is masked with one fixed group mask, so its nonzero count should be# identical across subjects. Compare against the mode, and ignore all-zero# volumes -- those are caught separately and would otherwise drag in every row.falff_dev = []for m in FALFF:    v = g[(g["modality"] == m) & (g["n_nonzero"] > 0)]    if len(v):        falff_dev.append(v[v["n_nonzero"] != v["n_nonzero"].mode().iat[0]])falff_dev = pd.concat(falff_dev) if falff_dev else g.iloc[0:0]checks = {    "NaNs present":             g[g["n_nan"] > 0],    "negative values":          g[g["n_neg"] > 0],    "all-zero volume":          g[g["n_nonzero"] == 0],    "probseg outside [0,1]":    g[g["modality"].isin(PROBSEG) & (g["vmax"] > 1.0 + 1e-5)],    "fALFF mask size off-mode": falff_dev,}for name, df in checks.items():    print(f"{name:28s} {len(df):5d}" + ("   <-- investigate" if len(df) else ""))    if len(df):        display(df[["subject_id","modality","n_nonzero","vmin","vmax"]].head(5))print()for m in FALFF:    v = g.loc[(g["modality"] == m) & (g["n_nonzero"] > 0), "n_nonzero"]    if len(v):        print(f"{m}: nonzero {int(v.min())}..{int(v.max())} over {len(v)} non-empty "              f"({'constant - good' if v.min() == v.max() else 'VARIES - group mask broke'})")

## 5. OutliersMedian/MAD z-scores rather than mean/std, so a few broken subjects cannot inflate thescale and hide inside it.

In [ ]:
Z_THRESH = 5.0METRICS  = ["mass", "mean_nz", "std_nz", "n_nonzero"]z = qc[qc["ok"]].copy()for m in METRICS:    z[f"z_{m}"] = z.groupby("modality")[m].transform(robust_z)z["z_max"] = z[[f"z_{m}" for m in METRICS]].abs().max(axis=1)flagged = z[z["z_max"] > Z_THRESH].sort_values("z_max", ascending=False)print(f"{len(flagged)} subject-modality pairs with |robust z| > {Z_THRESH} "      f"({flagged['subject_id'].nunique()} distinct subjects)")display(flagged[["subject_id","modality","mass","mean_nz","n_nonzero","z_max"]].head(20))

In [ ]:
fig, axes = plt.subplots(len(METRICS), len(MODALITIES),                         figsize=(3.0*len(MODALITIES), 2.3*len(METRICS)), squeeze=False)for r, met in enumerate(METRICS):    for c, mod in enumerate(MODALITIES):        ax = axes[r][c]        ax.hist(z.loc[z["modality"] == mod, met].dropna(), bins=60,                color="#4C72B0", edgecolor="none")        for x in flagged.loc[flagged["modality"] == mod, met]:            ax.axvline(x, color="#C44E52", lw=0.8, alpha=0.7)        if r == 0: ax.set_title(mod, fontsize=9)        if c == 0: ax.set_ylabel(met, fontsize=9)        ax.tick_params(labelsize=7); ax.set_yticks([])fig.suptitle(f"QC distributions; red = |robust z| > {Z_THRESH}", fontsize=10)fig.tight_layout(); plt.show()

## 6. One subjectSet `SID` to anything section 5 flagged.

In [ ]:
SID = subjects[0]fig, axes = plt.subplots(len(MODALITIES), 3, figsize=(8, 2.3*len(MODALITIES)))for r, mod in enumerate(MODALITIES):    if not out_path(SID, mod).exists():        for ax in axes[r]: ax.axis("off")        axes[r][1].text(.5, .5, f"{mod} missing", ha="center"); continue    ortho(axes[r], load(SID, mod), mod, vmax=1.0 if mod in PROBSEG else None)for ax, lab in zip(axes[0], ["sagittal", "coronal", "axial"]):    ax.set_title(lab, fontsize=9)fig.suptitle(SID, fontsize=11); fig.tight_layout(); plt.show()

## 7. Group meansA group mean that still looks like a brain is strong evidence alignment held acrosssubjects: if any subset were mirrored or shifted, the mean would ghost or blur.

In [ ]:
N_SAMPLE = 100rng = np.random.default_rng(0)sample = list(rng.choice(subjects, size=min(N_SAMPLE, len(subjects)), replace=False))means, skipped_shape = {}, []for mod in MODALITIES:    acc, n = np.zeros(EXPECTED_SHAPE, dtype=np.float64), 0    for s in sample:        p = out_path(s, mod)        if not p.exists():            continue        v = load(s, mod)        if v.shape != EXPECTED_SHAPE:      # a single odd file must not kill the average            skipped_shape.append((s, mod, v.shape)); continue        acc += v; n += 1    means[mod] = acc / max(n, 1)    print(f"{mod:16s} averaged {n:4d} subjects")if skipped_shape:    print(f"\nskipped {len(skipped_shape)} files with the wrong shape:")    for s, m, sh in skipped_shape[:10]:        print(f"  {s} {m} {sh}")fig, axes = plt.subplots(len(MODALITIES), 3, figsize=(8, 2.3*len(MODALITIES)))for r, mod in enumerate(MODALITIES):    ortho(axes[r], means[mod], f"mean {mod}", cmap="magma",          vmax=1.0 if mod in PROBSEG else None)fig.suptitle(f"group mean, n={len(sample)}", fontsize=11)fig.tight_layout(); plt.show()

## 8. Cross-modality alignment  *(needs section 7)*The real test of the resample: do GM and fALFF land in the *same voxels*? A mirror or ashift shows up here and almost nowhere else.

In [ ]:
if "means" not in globals():    raise RuntimeError("run section 7 first -- this section uses its group means")gm, fa = means["GM_probseg"], means["fALFF"]k = EXPECTED_SHAPE[2] // 2gm_m, fa_m = gm > 0.3, fa != 0denom = gm_m.sum() + fa_m.sum()dice = 2*(gm_m & fa_m).sum() / denom if denom else float("nan")fig, axes = plt.subplots(1, 3, figsize=(11, 3.6))axes[0].imshow(fa[:,:,k].T, origin="lower", cmap="magma")axes[0].contour(gm[:,:,k].T, levels=[0.3], colors="#4FD1C5", linewidths=0.8)axes[0].set_title("mean fALFF + GM 0.3 contour", fontsize=9)axes[1].imshow(gm_m[:,:,k].T.astype(float) - fa_m[:,:,k].T.astype(float),               origin="lower", cmap="coolwarm", vmin=-1, vmax=1)axes[1].set_title("GM>0.3 minus fALFF support\nred = GM only, blue = fALFF only", fontsize=9)axes[2].plot([2*(gm_m[:,:,zz] & fa_m[:,:,zz]).sum() /              max(gm_m[:,:,zz].sum() + fa_m[:,:,zz].sum(), 1)              for zz in range(EXPECTED_SHAPE[2])], color="#4C72B0")axes[2].set_ylim(0, 1); axes[2].set_xlabel("axial slice", fontsize=8)axes[2].set_title(f"per-slice Dice\nvolume Dice = {dice:.3f}", fontsize=9)for a in axes[:2]: a.set_xticks([]); a.set_yticks([])fig.tight_layout(); plt.show()print(f"Dice(GM>0.3, fALFF support) = {dice:.3f}")print("  expect roughly 0.6-0.8. Much lower means the modalities are not landing in the")print("  same voxels -- check the affines before trusting any fusion result.")

## 9. Verdict

In [ ]:
# Defensive: reports on whichever sections were actually run.G = globals()problems, skipped = [], []def seen(name):    if name in G: return True    skipped.append(name); return Falseif seen("missing_chunks") and missing_chunks: problems.append(f"{len(missing_chunks)} chunk logs absent {missing_chunks}")if seen("gaps") and len(gaps):                problems.append(f"{len(gaps)} subjects with no log row")if seen("bad") and len(bad):                  problems.append(f"{len(bad)} non-success log rows")if seen("inv") and inv['missing'].sum():      problems.append(f"{int(inv['missing'].sum())} missing files")if seen("extra") and len(extra):              problems.append(f"{len(extra)} unexpected files")if seen("stray") and len(stray):              problems.append(f"{len(stray)} stray .part files")if seen("sig") and len(sig) > 1:              problems.append(f"{len(sig)} distinct header signatures")if seen("qc") and (~qc['ok']).sum():          problems.append(f"{int((~qc['ok']).sum())} unreadable files")if seen("checks"):    for name, df in checks.items():        if len(df):                           problems.append(f"{len(df)} x {name}")if seen("flagged") and len(flagged):          problems.append(f"{len(flagged)} statistical outliers (review, not necessarily broken)")if seen("dice") and dice < 0.5:               problems.append(f"GM/fALFF Dice only {dice:.3f} -- possible misalignment")print(f"{len(subjects)} subjects x {len(MODALITIES)} modalities = "      f"{len(subjects)*len(MODALITIES)} expected\n")if problems:    print("ISSUES:")    for p in problems: print(f"  - {p}")else:    print("No issues found in the sections that were run.")if skipped:    print(f"\nnot checked (section not run): {sorted(set(skipped))}")